<a href="https://colab.research.google.com/github/jojojojo-jo/LRP_analysis_resnet/blob/main/FGSM_attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#0. Installations

In [ ]:
!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0

ERROR: Operation cancelled by user


In [ ]:
print(torch.__version__)

2.9.0+cu128


In [ ]:
!pip install zennit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.2 MB/s eta 0:00:00


In [ ]:
import torch
from torch import nn
from torch.nn import Linear
import torch.nn.functional as F

from itertools import islice

from PIL import Image

import torchvision
from torchvision import datasets, models, transforms
from torchvision.transforms import Compose, Resize, CenterCrop
from torchvision.transforms import ToTensor, Normalize
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

from zennit.attribution import Gradient, SmoothGrad
from zennit.core import Stabilizer
from zennit.composites import EpsilonGammaBox, EpsilonPlusFlat
from zennit.composites import SpecialFirstLayerMapComposite, NameMapComposite
from zennit.image import imgify, imsave
from zennit.rules import Epsilon, ZPlus, ZBox, Norm, Pass, Flat
from zennit.types import Convolution, Activation, AvgPool, Linear as AnyLinear
from zennit.types import BatchNorm
from zennit.torchvision import ResNetCanonizer

import os
import numpy as np
import matplotlib.pyplot as plt

#import torchinfo.summary
try:
  from torchinfo import summary
except:
  print("[INFO] Couldn't find torchinfo... installing it")
  !pip install -q torchinfo
  from torchinfo import summary

#import helper functions
try:
  from going_modular.going_modular import data_setup, engine
except:
  print(f"[INFO] Couldn't find going_modular scripts... downloading from GitHub")
  !git clone https://github.com/mrdbourke/pytorch-deep-learning
  !mv pytorch-deep-learning/going_modular .
  !rm -rf pytorch-deep-learning
  from going_modular.going_modular import data_setup, engine

[INFO] Couldn't find torchinfo... installing it
[INFO] Couldn't find going_modular scripts... downloading from GitHub
Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 4410, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 4410 (delta 5), reused 2 (delta 2), pack-reused 4400 (from 2)
Receiving objects: 100% (4410/4410), 764.18 MiB | 17.46 MiB/s, done.
Resolving deltas: 100% (2661/2661), done.
Updating files: 100% (248/248), done.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

#1. Get data

In [ ]:
import os
import zipfile

from pathlib import Path

import requests

data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi_20_percent"

if image_path.is_dir():
  print(f"{image_path} directory exists")
else:
  print(f"Did not find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)

  with open(data_path / "pizza_steak_sushi_20_percent.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip")
    print("Downloading data...")
    f.write(request.content)

  with zipfile.ZipFile(data_path / "pizza_steak_sushi_20_percent.zip", "r") as zip_ref:
    print("Unzipping data...")
    zip_ref.extractall(image_path)

  os.remove(data_path / "pizza_steak_sushi_20_percent.zip")

Did not find data/pizza_steak_sushi_20_percent directory, creating one...
Unzipping data...


In [ ]:
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(PosixPath('data/pizza_steak_sushi_20_percent/train'),
 PosixPath('data/pizza_steak_sushi_20_percent/test'))

In [ ]:
from numpy.matrixlib import test

weights = torchvision.models.ResNet18_Weights.DEFAULT
auto_transforms = weights.transforms()
print(f"Automatically created transforms: {auto_transforms}")

train_dataset = datasets.ImageFolder(train_dir, transform=auto_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=auto_transforms)

# Create dataloaders with batch_size=32 for normal training/testing
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms,
                                                                               batch_size=32)

# Create a separate test_dataloader with batch_size=1 for the FGSM attack function
# because the original 'test' function iterates over items and uses .item()
_, test_dataloader_fgsm, _ = data_setup.create_dataloaders(train_dir=train_dir,
                                                           test_dir=test_dir,
                                                           transform=auto_transforms,
                                                           batch_size=1) # Batch size 1 for FGSM test function

train_dataloader, test_dataloader, class_names, test_dataloader_fgsm

Automatically created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


(<torch.utils.data.dataloader.DataLoader at 0x7a41fd168ef0>,
 ['pizza', 'steak', 'sushi'],
 <torch.utils.data.dataloader.DataLoader at 0x7a4204770aa0>)

#2. Load model

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
model = models.resnet18(weights=weights)

num_classes = 3
model.fc = nn.Linear(model.fc.in_features, num_classes)

model_path = '/content/drive/MyDrive/study/Project/models/best_model_plateau.pth'
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

model.to(device) # Move the model to the correct device (GPU in this case)
model.eval()

print("Model loaded successfully!")

Model loaded successfully!


#3. Implement FGSM attack

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
epsilons = [0, .05, .1, .15, .2, .25, .3]

# Set random seed for reproducibility
torch.manual_seed(42)

# FGSM attack code
def fgsm_attack(image, epsilon, data_grad):
    # Collect the element-wise sign of the data gradient
    sign_data_grad = data_grad.sign()
    # Create the perturbed image by adjusting each pixel of the input image
    perturbed_image = image + epsilon*sign_data_grad
    # Adding clipping to maintain [0,1] range
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    # Return the perturbed image
    return perturbed_image

# restores the tensors to their original scale
def denorm(batch, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """
    Convert a batch of tensors to their original scale.

    Args:
        batch (torch.Tensor): Batch of normalized tensors.
        mean (torch.Tensor or list): Mean used for normalization.
        std (torch.Tensor or list): Standard deviation used for normalization.

    Returns:
        torch.Tensor: batch of tensors without normalization applied to them.
    """
    if isinstance(mean, list):
        mean = torch.tensor(mean).to(device)
    if isinstance(std, list):
        std = torch.tensor(std).to(device)

    return batch * std.view(1, -1, 1, 1) + mean.view(1, -1, 1, 1)

In [ ]:
def test(model, device, test_loader, epsilon ):

    # Accuracy counter
    correct = 0
    adv_examples = []

    # Define normalization constants for ResNet18 pre-trained on ImageNet.
    # These should be moved to the device to avoid type mismatches.
    normalize_mean_tensor = torch.tensor([0.485, 0.456, 0.406]).to(device).view(1, 3, 1, 1)
    normalize_std_tensor = torch.tensor([0.229, 0.224, 0.225]).to(device).view(1, 3, 1, 1)

    # Loop over all examples in test set
    total_images_processed = 0 # To count total images for accuracy
    for data, target in test_loader: # test_loader here is expected to have batch_size=1

        total_images_processed += 1 # Increment total image count for accuracy denominator

        # Send the data and label to the device
        data, target = data.to(device), target.to(device)

        # Set requires_grad attribute of tensor. Important for Attack
        data.requires_grad = True

        # Forward pass the data through the model
        output = model(data)
        init_pred = output.max(1, keepdim=True)[1] # get the index of the max log-probability

        # If the initial prediction is wrong, don't bother attacking, just move on
        if init_pred.item() != target.item():
            continue

        # Calculate the loss
        loss = F.nll_loss(output, target)

        # Zero all existing gradients
        model.zero_grad()

        # Calculate gradients of model in backward pass
        loss.backward()

        # Collect ``datagrad``
        data_grad = data.grad.data

        # Restore the data to its original scale (0-1 range)
        # denorm function now correctly uses the ImageNet mean/std
        data_denorm = denorm(data,
                             mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])

        # Call FGSM Attack
        perturbed_data = fgsm_attack(data_denorm, epsilon, data_grad)

        # Reapply normalization with correct mean/std tensors on device
        perturbed_data_normalized = (perturbed_data - normalize_mean_tensor) / normalize_std_tensor

        # Re-classify the perturbed image
        output = model(perturbed_data_normalized)

        # Check for success
        final_pred = output.max(1, keepdim=True)[1] # get the index of the max log-probability
        if final_pred.item() == target.item():
            correct += 1
            # Special case for saving 0 epsilon examples
            if epsilon == 0 and len(adv_examples) < 5:
                adv_ex = perturbed_data.squeeze().detach().cpu().numpy()
                adv_examples.append( (init_pred.item(), final_pred.item(), adv_ex) )
        else:
            # Save some adv examples for visualization later
            if len(adv_examples) < 5:
                adv_ex = perturbed_data.squeeze().detach().cpu().numpy()
                adv_examples.append( (init_pred.item(), final_pred.item(), adv_ex) )

    # Calculate final accuracy for this epsilon
    final_acc = correct / float(total_images_processed) if total_images_processed > 0 else 0.0
    print(f"Epsilon: {epsilon}\tTest Accuracy = {correct} / {total_images_processed} = {final_acc}")

    # Return the accuracy and an adversarial example
    return final_acc, adv_examples

In [ ]:
accuracies = []
examples = []

# Run test for each epsilon
for eps in epsilons:
    acc, ex = test(model, device, test_dataloader_fgsm, eps) # Use the new dataloader with batch_size=1
    accuracies.append(acc)
    examples.append(ex)

Epsilon: 0	Test Accuracy = 140 / 150 = 0.9333333333333333
Epsilon: 0.05	Test Accuracy = 19 / 150 = 0.12666666666666668
Epsilon: 0.1	Test Accuracy = 42 / 150 = 0.28
Epsilon: 0.15	Test Accuracy = 45 / 150 = 0.3
Epsilon: 0.2	Test Accuracy = 44 / 150 = 0.29333333333333333
Epsilon: 0.25	Test Accuracy = 42 / 150 = 0.28
Epsilon: 0.3	Test Accuracy = 44 / 150 = 0.29333333333333333


In [ ]:
plt.figure(figsize=(5,5))
plt.plot(epsilons, accuracies, "*-")
plt.yticks(np.arange(0, 1.1, step=0.1))
plt.xticks(np.arange(0, .35, step=0.05))
plt.title("Accuracy vs Epsilon")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# Plot several examples of adversarial samples at each epsilon
cnt = 0
plt.figure(figsize=(8,10))
for i in range(len(epsilons)):
    for j in range(len(examples[i])):
        cnt += 1
        plt.subplot(len(epsilons),len(examples[0]),cnt)
        plt.xticks([], [])
        plt.yticks([], [])
        if j == 0:
            plt.ylabel(f"Eps: {epsilons[i]}", fontsize=14)
        orig,adv,ex = examples[i][j]
        plt.title(f"{orig} -> {adv}")
        plt.imshow(ex, cmap="gray")
plt.tight_layout()
plt.show()